In [1]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'gdown'])
import urllib.request
urllib.request.urlopen('http://google.com')
print("✅ Internet works")
import os, torch
print(f"✅ GPU: {os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
print(f"✅ Disk: {os.popen('df -h /kaggle/working').read().strip()}")

✅ Internet works
✅ GPU: Tesla T4
Tesla T4
✅ CUDA: True
✅ Disk: Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   80K   20G   1% /kaggle/working


In [2]:
import os
os.system('pip install -q PyDrive2')
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
print("✅ Drive authenticated")

os.system('git clone https://github.com/nabirakhan/luxe /kaggle/working/Luxe')
print("✅ Repo cloned")

with open('/kaggle/working/Luxe/backend/pgd_modification.py', 'r') as f:
    content = f.read()
content = content.replace(
    'clip_model, _ = clip.load("ViT-L/14", device=self._device)',
    'clip_model, _ = clip.load("ViT-B/32", device=self._device)'
)
with open('/kaggle/working/Luxe/backend/pgd_modification.py', 'w') as f:
    f.write(content)
print("✅ Bug fix applied")

✅ Drive authenticated


Cloning into '/kaggle/working/Luxe'...


✅ Repo cloned
✅ Bug fix applied


In [3]:
os.makedirs('/kaggle/working/Luxe/backend/checkpoints', exist_ok=True)
checkpoints = {
    'segformer_lip.pth':  '1-moZPPGjOPrk3zoSKhnePQi0DQLIQqxn',
    'sd_inpaint_vae.pth': '11CQk-b1Mz9hmCzpUPEf2XJ80Yjr8sLHj',
    'ipp_vae.pth':        '1ZW8xnXiJppebAFH3K0bvOfPbYMSNocai',
    'ip_adapter.pth':     '1ctOYbFcwAAwWvbJqA1MV2Ou21NBkimNX',
}
for fname, fid in checkpoints.items():
    out = f'/kaggle/working/Luxe/backend/checkpoints/{fname}'
    f = drive.CreateFile({'id': fid})
    f.GetContentFile(out)
    print(f"✅ {fname}")
print(f"\nDisk: {os.popen('df -h /kaggle/working').read().strip()}")

✅ segformer_lip.pth
✅ sd_inpaint_vae.pth
✅ ipp_vae.pth
✅ ip_adapter.pth

Disk: Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  768M   19G   4% /kaggle/working


In [4]:
import zipfile
os.makedirs('/kaggle/working/deepfashion/img', exist_ok=True)
os.makedirs('/kaggle/working/unet_pairs', exist_ok=True)

print("Downloading img.zip...")
f = drive.CreateFile({'id': '1VDmSzYS4jeH-45c7JMVF22bTrqzdW0Gp'})
f.GetContentFile('/kaggle/working/deepfashion/img/img.zip')
print("Extracting...")
with zipfile.ZipFile('/kaggle/working/deepfashion/img/img.zip', 'r') as z:
    z.extractall('/kaggle/working/deepfashion/img/')
os.remove('/kaggle/working/deepfashion/img/img.zip')
print("✅ img extracted")

f = drive.CreateFile({'id': '115QWWX_yttJYICGuvHw7wAf6y7Ahj1P3'})
f.GetContentFile('/kaggle/working/deepfashion/list_eval_partition.txt')
print("✅ partition file downloaded")

from pathlib import Path
with open('/kaggle/working/deepfashion/list_eval_partition.txt') as f:
    lines = f.readlines()[2:]
image_names = [l.strip().split()[0] for l in lines
               if len(l.strip().split()) >= 3 and l.strip().split()[2] == 'train']
found = sum(1 for img in image_names[:500]
            if (Path('/kaggle/working/deepfashion/img') / img).exists())
print(f"✅ Image path check: {found}/500 found")
print(f"Disk: {os.popen('df -h /kaggle/working').read().strip()}")

Extracting...
✅ img extracted
✅ partition file downloaded
✅ Image path check: 500/500 found
Disk: Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  1.7G   18G   9% /kaggle/working


In [5]:
os.system('pip install -q diffusers accelerate transformers lpips')
os.system('pip install -q git+https://github.com/openai/CLIP.git')
print("✅ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
✅ Dependencies installed


In [6]:
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
UNET_PAIRS_FOLDER_ID = '15-K1jg_MWqj6J0FJCDoaGm9aeKc0tgoC'
MAX_PAIRS = 3000

import sys, logging
from pathlib import Path
import torch
from tqdm.auto import tqdm

sys.path.insert(0, '/kaggle/working/Luxe/backend')
logging.basicConfig(level=logging.WARNING)

import config
config.STEPS_PGD       = 40
config.STEPS_CLIP      = 40
config.LPIPS_THRESHOLD = 0.20   # ← ADD THIS LINE

from protect import Protector
protector = Protector()

LOCAL_PAIRS = Path('/kaggle/working/unet_pairs')
LOCAL_PAIRS.mkdir(exist_ok=True)

def get_drive():
    from pydrive2.auth import GoogleAuth
    from pydrive2.drive import GoogleDrive
    from google.colab import auth
    from oauth2client.client import GoogleCredentials
    auth.authenticate_user()
    gauth = GoogleAuth()
    gauth.credentials = GoogleCredentials.get_application_default()
    return GoogleDrive(gauth)

def img_name_to_filename(img_name):
    parts = Path(img_name).parts
    rel = parts[1:]  # skip leading 'img'
    return '_'.join(rel).replace('.jpg', '.pt')

def upload_pair_to_drive(local_path, filename, folder_id, max_attempts=3):
    global drive
    for attempt in range(max_attempts):
        try:
            file_list = drive.ListFile({
                'q': f"title='{filename}' and '{folder_id}' in parents and trashed=false"
            }).GetList()
            if file_list:
                f = drive.CreateFile({'id': file_list[0]['id']})
            else:
                f = drive.CreateFile({'title': filename, 'parents': [{'id': folder_id}]})
            f.SetContentFile(str(local_path))
            f.Upload()
            verify = drive.ListFile({
                'q': f"title='{filename}' and '{folder_id}' in parents and trashed=false"
            }).GetList()
            if verify:
                return True
            print(f"  ⚠️ Upload unverified, retrying ({attempt+1})...")
        except Exception as e:
            print(f"  🔄 Reauthenticating (attempt {attempt+1}): {e}")
            drive = get_drive()
    return False

def upload_local_backlog(folder_id):
    local_files = list(LOCAL_PAIRS.glob('*.pt'))
    if not local_files:
        return 0
    print(f"  📤 Uploading backlog of {len(local_files)} local files...")
    uploaded = 0
    for local_path in local_files:
        success = upload_pair_to_drive(local_path, local_path.name, folder_id)
        if success:
            os.remove(local_path)
            uploaded += 1
        else:
            print(f"  ❌ Backlog upload failed for {local_path.name} — keeping locally")
    print(f"  ✅ Backlog: {uploaded}/{len(local_files)} uploaded")
    return uploaded

def get_already_generated(folder_id):
    global drive
    try:
        file_list = drive.ListFile({
            'q': f"'{folder_id}' in parents and trashed=false"
        }).GetList()
        return {f['title'] for f in file_list}
    except Exception as e:
        print(f"⚠️ Could not fetch existing pairs: {e}")
        drive = get_drive()
        return set()

# Load image list
with open('/kaggle/working/deepfashion/list_eval_partition.txt') as f:
    lines = f.readlines()[2:]
image_names = [l.strip().split()[0] for l in lines
               if len(l.strip().split()) >= 3 and l.strip().split()[2] == 'train']
print(f"Total train images: {len(image_names)}")

filenames = [img_name_to_filename(img) for img in image_names]
print(f"Unique filenames: {len(set(filenames))}/{len(filenames)}")

upload_local_backlog(UNET_PAIRS_FOLDER_ID)
print("Checking Drive for already generated pairs...")
already_done = get_already_generated(UNET_PAIRS_FOLDER_ID)
print(f"Already on Drive: {len(already_done)} pairs — skipping these")

generated     = 0
skipped       = 0
failed_upload = 0

pbar = tqdm(image_names, desc="Generating pairs")
for img_name in pbar:
    if generated + len(already_done) >= MAX_PAIRS:
        print(f"🎉 Reached {MAX_PAIRS} pairs target!")
        break

    filename = img_name_to_filename(img_name)
    if filename in already_done:
        skipped += 1
        pbar.set_postfix({'gen': generated, 'skip': skipped, 'fail': failed_upload})
        continue

    img_path = Path('/kaggle/working/deepfashion/img') / img_name
    if not img_path.exists():
        continue

    try:
        with open(img_path, 'rb') as f:
            image_bytes = f.read()
        x_orig, delta, mask = protector.protect_pgd_only(image_bytes, mode='full')
        local_path = LOCAL_PAIRS / filename
        torch.save({'x_orig': x_orig, 'delta': delta, 'mask': mask}, local_path)
        success = upload_pair_to_drive(local_path, filename, UNET_PAIRS_FOLDER_ID)
        if success:
            os.remove(local_path)
            already_done.add(filename)
            generated += 1
        else:
            failed_upload += 1
        pbar.set_postfix({'gen': generated, 'skip': skipped,
                          'fail': failed_upload,
                          'total': generated + len(already_done)})
        if generated % 10 == 0 and generated > 0:
            print(f"  ✅ {generated} new | {len(already_done)} total on Drive | {failed_upload} pending locally")
        if generated % 50 == 0 and generated > 0:
            upload_local_backlog(UNET_PAIRS_FOLDER_ID)
            already_done = get_already_generated(UNET_PAIRS_FOLDER_ID)
    except Exception as e:
        print(f"  ❌ Failed on {img_name}: {e}")
        torch.cuda.empty_cache()
        continue

print("\nFinal backlog flush...")
upload_local_backlog(UNET_PAIRS_FOLDER_ID)
final_count = get_already_generated(UNET_PAIRS_FOLDER_ID)
print(f"\n🎉 Session done! Generated={generated} | Total on Drive={len(final_count)} | Failed={failed_upload}")

preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/110M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

...

  ⚠️ Upload unverified, retrying (1)...


  ⚠️ Upload unverified, retrying (1)...


  ⚠️ Upload unverified, retrying (1)...


  ⚠️ Upload unverified, retrying (1)...
  ✅ 240 new | 2601 total on Drive | 0 pending locally


KeyboardInterrupt: 